# 02. Exploratory Data Analysis & System Load Dynamics

## Electrical Engineering Context
Distribution networks exhibit pronounced diurnal, seasonal, and load-dependent behavioral profiles. In this notebook, we explore:
* **Diurnal load cycles:** Industrial 3-shift operations vs residential dual-peaks (morning/evening) vs commercial business hours.
* **Ambient temperature correlation:** Summer cooling loads inflating active power demand.
* **Electrical correlation matrix:** Relationships between $V, I, P, Q, S, PF, T_{amb}, T_{equip}$.
* **Transformer coincident demand:** Aggregated loading across substations.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.utils.config import PROCESSED_DATA_DIR

df_meas = pd.read_csv(PROCESSED_DATA_DIR / "fact_electrical_measurements.csv")
dim_fdr = pd.read_csv(PROCESSED_DATA_DIR / "dim_feeder.csv")
df_meas["timestamp"] = pd.to_datetime(df_meas["timestamp"])
print(f"Loaded {len(df_meas):,} clean records.")


## 1. Diurnal Load Profiles by Load Category
We analyze hourly active power curves to observe shift work, commercial peaks, and domestic evening surges.


In [ ]:
df_meas["hour"] = df_meas["timestamp"].dt.hour
df_merged = df_meas.merge(dim_fdr[["feeder_id", "load_category"]], on="feeder_id")

diurnal = df_merged.groupby(["load_category", "hour"])["active_power_kw"].mean().reset_index()

plt.figure(figsize=(12, 5.5))
for cat, grp in diurnal.groupby("load_category"):
    plt.plot(grp["hour"], grp["active_power_kw"], marker="o", label=cat, linewidth=2)
plt.title("Diurnal Active Power Profile by Industrial / Municipal Load Category", fontsize=13, fontweight="bold")
plt.xlabel("Hour of Day (00:00 - 23:00)", fontsize=11)
plt.ylabel("Mean Active Demand (kW)", fontsize=11)
plt.xticks(range(0, 24, 2))
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 2. Electrical Quantities Correlation Heatmap
We examine cross-correlations across voltages, currents, active/reactive powers, power factor, and temperatures.


In [ ]:
corr_cols = [
    "voltage_avg", "current_avg", "active_power_kw", "reactive_power_kvar",
    "apparent_power_kva", "power_factor", "feeder_loss_kw", "feeder_loss_pct",
    "ambient_temperature_c", "equipment_temperature_c"
]
corr_matrix = df_meas[corr_cols].corr()

plt.figure(figsize=(10, 7.5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Matrix of Electrical & Thermal Quantities", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Transformer Loading Distribution
We evaluate how transformer loading percentages vary across the fleet.


In [ ]:
plt.figure(figsize=(10, 4.5))
sns.boxplot(data=df_meas, x="transformer_id", y="transformer_loading_pct", palette="Set2")
plt.axhline(100.0, color="red", linestyle="--", label="100% Rated Capacity")
plt.axhline(80.0, color="orange", linestyle=":", label="80% Alert Threshold")
plt.title("Transformer Loading % Distribution Across 6 Months", fontsize=12, fontweight="bold")
plt.xlabel("Transformer ID")
plt.ylabel("Loading (% MVA)")
plt.legend()
plt.tight_layout()
plt.show()
